# Phase 4 - Notebook 04: LIVE DEMO - train the models and extract attributes in a GUI

**For the viva.** Accelerator = GPU T4 (x1 is enough), Internet = On. Total ~35-40 min (CRF 2 min, BERT ~10 min, DeBERTa ~20 min).
The notebook (1) rebuilds the BIO data from raw WDC-PAVE, (2) trains the classical CRF, (3) fine-tunes BERT and DeBERTa-v3
with live per-epoch loss and validation F1, (4) scores everything plus the CPEC committee on the official test set and
(5) opens a Gradio GUI (public link) where any product title/description can be typed in.
For a short demo set `EPOCHS = 3` (~8 min); the Phase 2 numbers use 14/15 epochs.

In [ ]:
# ---- 1. Get the code: clone the project repository (Internet must be ON) ----
REPO_URL = "https://github.com/REPLACE_WITH_YOUR_USERNAME/nlp-product-attribute-extraction.git"
import os, subprocess, sys
from pathlib import Path
ROOT = Path("/kaggle/working/nlp-product-attribute-extraction")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT); sys.path[:0] = [str(ROOT)]
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sklearn-crfsuite", "seqeval", "gradio", "sentencepiece"], check=True)
EPOCHS = 15          # 3 for a quick demo
!python scripts/data/build_wdc_bio.py

## 1. Classical baseline: CRF (train on 906 products, score on validation)

In [ ]:
!python scripts/phase2_baselines/train_crf.py --mode dev --out models/crf_dev
!cp models/crf_dev/model.pkl models/crf.pkl

## 2. Modern baselines: fine-tune BERT and DeBERTa-v3 (live training log, validation F1 per epoch)

In [ ]:
!python scripts/phase2_baselines/train_transformer.py --model bert --mode dev --epochs $EPOCHS --out models/bert

In [ ]:
!python scripts/phase2_baselines/train_transformer.py --model deberta --mode dev --epochs $EPOCHS --out models/deberta

## 3. Score CRF / BERT / DeBERTa / CPEC committee on the official 354-product test set

In [ ]:
import json, pickle, sys
from collections import Counter
from src.attrext.io import read_jsonl
from src.attrext.crf import predict_crf
from src.attrext.transformer_tagger import TransformerTagger
from src.attrext.committee import committee_predict, score_rows
test = read_jsonl("datasets/wdc_pave/bio/test.jsonl")
freq = Counter(l[2:] for r in read_jsonl("datasets/wdc_pave/bio/train.jsonl") for l in r["labels"] if l.startswith("B-"))
json.dump(freq, open("models/train_span_counts.json", "w"))
def rows(pred): return [dict(id=r["id"], tokens=r["tokens"], gold=r["labels"], predicted=p) for r, p in zip(test, pred)]
systems = {"CRF": rows(predict_crf(pickle.load(open("models/crf.pkl", "rb")), test)),
           "BERT": rows(TransformerTagger(None, checkpoint="models/bert").predict(test)),
           "DeBERTa": rows(TransformerTagger(None, checkpoint="models/deberta").predict(test))}
tau = json.load(open("results/phase3_committee/wdc/results.json"))["selected_thresholds"]
systems["CPEC committee"] = committee_predict([systems["DeBERTa"], systems["BERT"], systems["CRF"]], freq, tau)
print(f"{'system':<16}{'P':>8}{'R':>8}{'micro-F1':>10}")
for k, v in systems.items():
    s = score_rows(v); print(f"{k:<16}{s['precision']*100:8.2f}{s['recall']*100:8.2f}{s['micro_f1']*100:10.2f}")

## 4. CLI and GUI on raw text (annotation-free tokenisation)

In [ ]:
!python demo/cli.py --models models --freq models/train_span_counts.json --title "Xerox TONER CARTRIDGE FOR WORKCENTRE 4250 - 106R01409" --description "25K PAGE YIELD COLOR: Black UPC: 095205742466"

In [ ]:
from demo.predictor import Extractor
from demo.app import build
ui = build(Extractor("models", json.load(open("models/train_span_counts.json")), tau))
ui.launch(share=True)   # click the public *.gradio.live link